![Databricks Academy](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/db-academy.png)

# 10 - Deploying a Pipeline to Production

In this demonstration, we will begin by adding an additional data source to our pipeline and performing a join with our streaming tables. Then, we will focus on productionizing the pipeline by adding comments and table properties to the objects we create, scheduling the pipeline, and creating an event log to monitor the pipeline.

### Learning Objectives

By the end of this lesson, you will be able to:
- Apply the appropriate comment syntax and table properties to pipeline objects to enhance readability.
- Demonstrate how to perform a join between two streaming tables using a materialized view to optimize data processing.
- Execute the scheduling of a pipeline using trigger or continuous modes to ensure timely processing.
- Explore the event log to monitor a production Apache Spark™ Declarative Pipeline.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

1. Run the following cell to configure your working environment for this course.

    This cell will also reset your `/Volumes/labuser/sdp_1_bronze/source` volume with the JSON files to the starting point, with one JSON file in each directory.

In [0]:
%run ./Includes/Classroom-Setup-REQUIRED

## B. Explore the Orders and Status JSON Files

1. Explore the raw data located in the `/Volumes/labuser/sdp_1_bronze/source/orders/` volume.

   This is the data we have been working with throughout the course demonstrations.

   Run the cell below to view the results. Notice that the orders JSON file(s) contains information about when each order was placed.

In [0]:
SELECT *
FROM read_files(
  source_volume_path || '/orders/',
  format => 'JSON'
)
LIMIT 10;

2. Explore the **status** raw data located in the `/Volumes/labuser/sdp_1_bronze/source/status/` volume and filter for the specific **order_id** *75123*.

   Run the cell below to view the results. Notice that the status JSON file(s) contain **order_status** information for each order.

   **NOTE:** The **order_status** can include multiple rows per order and may be any of the following:

   - on the way
   - canceled
   - return canceled
   - reported shipping error
   - delivered
   - return processed
   - return picked up
   - placed
   - preparing
   - return requested


In [0]:
SELECT *
FROM read_files(
  source_volume_path || '/status/',
  format => 'JSON'
)
WHERE order_id = 75123;

3. One of our objectives is to join the **orders** data with the order **status** data.

    The query below demonstrates what the result of the final join in the Spark Declarative Pipeline will look like after the data has been incrementally ingested and cleaned when we create the pipeline. Run the cell and review the output.

    Notice that after joining the tables, we can see each **order_id** along with its original **order_timestamp** and the **order_status** at specific points in time.

**NOTE:** The data used in this demo is artificially generated, so the **order_status_timestamps** may not reflect realistic timing.

In [0]:
WITH orders AS (
  SELECT *
  FROM read_files(
        source_volume_path || '/orders/',
        format => 'JSON'
  )
),
status AS (
  SELECT *
  FROM read_files(
        source_volume_path || '/status/',
        format => 'JSON'
  )
)
-- Join the views to get the order history with status
SELECT
  orders.order_id,
  timestamp(orders.order_timestamp) AS order_timestamp,
  status.order_status,
  timestamp(status.status_timestamp) AS order_status_timestamp
FROM orders
  INNER JOIN status
  ON orders.order_id = status.order_id
ORDER BY order_id, order_status_timestamp;

## C. Putting a Pipeline in Production

This course includes a complete Apache Spark™ Declarative Pipeline project that has already been created.

In this section, you'll explore the Spark Declarative Pipeline and modify its settings for production use.


1. The screenshot below shows what the final Spark Declarative Pipeline will look like when ingesting a single JSON file from the data sources:
![Final Demo 6 Pipeline](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/deploying-a-pipeline-to-production/demo10_pipeline_image_run1.png)

    **Note:** Depending on the number of files you've ingested, the row count may vary.

2. Run the cell below to create your starter Spark Declarative Pipeline for this demonstration. The pipeline will set the following for you:
    - Your default catalog: **labuser**
    - Your configuration parameter: `source` = `/Volumes/labuser/sdp_1_bronze/source`

    **NOTE:** If the pipeline already exists, an error will be returned. In that case, you'll need to delete the existing pipeline and rerun this cell.

    To delete the pipeline:

    a. Select **Jobs & Pipelines** from the far-left navigation bar.

    b. Find the pipeline you want to delete.

    c. Click the three-dot menu ![ellipsis icon](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/ellipsis_icon.png).

    d. Select **Delete**.

**NOTE:**  The `create_declarative_pipeline` function is a custom function built for this course to create the sample pipeline using the Databricks REST API. This avoids manually creating the pipeline and referencing the pipeline assets.

In [0]:
%python
create_declarative_pipeline(
    pipeline_name=f'10 - Deploying a Pipeline to Production Project - {my_catalog}',
    root_path_folder_name='10 - Deploying a Pipeline to Production Project',
    catalog_name=my_catalog,
    schema_name='default',
    source_folder_names=['orders', 'status'],
    configuration={'source': source_volume_path}
)

3. Complete the following steps to open the starter Spark Declarative Pipeline project for this demonstration:

   a. In the main navigation bar, right-click on **Jobs & Pipelines** and select **Open Link in New Tab**.

   b. In **Jobs & Pipelines** select your **10 - Deploying a Pipeline to Production Project - labuser** pipeline.

   c. In the **Pipeline details** pane on the far right select **Open in Editor** (field to the right of **Source code**) to open the pipeline in the **Lakeflow Pipeline Editor**.

   d. In the new tab, you should see the following folders:
      - **explorations**
      - **orders**
      - **status**
      - plus the extra **python_excluded** folder that contains the Python version.

## D. Explore the code in the `orders/orders_pipeline.sql` file

1. In your **Spark Declarative Pipeline** select the **orders** folder.

2. This file contains the same **orders_pipeline.sql** pipeline you've been working with throughout the course.

3. Quickly review the code again. Notice the following:
    - Each streaming table or materialized view now includes a `COMMENT` and `TBLPROPERTIES` section to document each object.
    - Data quality expectations were added from the previous demonstration.

## E. Explore the code in the `status/status_pipeline` notebook

### E1. Open the `status/status_pipeline.sql` file
1. In your Spark Declarative Pipeline editor open the **status/status_pipeline.sql** file.

2. This file processes new data and adds it to the pipeline for order **status**.

### E2. Bronze Status Table Creation (`1_bronze_db.status_bronze_demo10`)

Use the code in section `A. Bronze Table Creation`, to understand how the Bronze streaming table is created and how data quality is enforced.

1. This statement creates the streaming table **1_bronze_db.status_bronze_demo10** by ingesting raw JSON files from your lab volume path: `/Volumes/labuser_/sdp_1_bronze/source/status/`

2. The `COMMENT` clause adds descriptive metadata to the table, making it easier to understand the table's purpose when browsing in Unity Catalog.

3. The `TBLPROPERTIES` section adds table level configuration:
   - `"quality" = "bronze"` labels this table as part of the Bronze layer in the medallion architecture.
   - `"pipelines.reset.allowed" = false` prevents full table refreshes, which helps avoid accidental truncation of the table and loss of checkpoints during pipeline resets.

4. The `${source}` variable in the `FROM STREAM` clause is used to dynamically reference your specific volume location using the SDP configuration parameter.

**NOTES**:
   - **Pipeline table properties**: [AWS](https://docs.databricks.com/aws/en/ldp/properties#pipeline-table-properties) |
   [Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/properties#pipeline-table-properties) |
   [GCP](https://docs.databricks.com/gcp/en/ldp/properties#pipeline-table-properties)

   - **For proper tagging visit Apply tags to Unity Catalog securable objects**: [AWS](https://docs.databricks.com/aws/en/database-objects/tags) |
   [Azure](https://learn.microsoft.com/en-us/azure/databricks/database-objects/tags) |
   [GCP](https://docs.databricks.com/gcp/en/database-objects/tags). 
      - Tagging is outside the scope of this course.

### E3. Silver Status Table Creation (`2_silver_db.status_silver_demo10`)

Use the code in section `B. Bronze -> Silver`, to understand how the Silver streaming table is created and how data quality is enforced.

1. This statement creates the streaming table **2_silver_db.status_silver_demo10** from the Bronze streaming table **1_bronze_db.status_bronze_demo10**.

2. The `SELECT` clause:
   - Selects only the required columns for the Silver layer.
   - Casts `status_timestamp` to a proper timestamp as `order_status_timestamp`.

3. The `CONSTRAINT` clauses define data quality expectations:
   - `valid_timestamp` drops rows where the timestamp is not valid.
   - `valid_order_status` warns when the status is not in the allowed list.

4. The `COMMENT` and `TBLPROPERTIES` document the table and label it as Silver in the medallion architecture.

### E4. Materialized View to Join Two Streaming Tables (`3_gold_db.full_order_info_gold_demo10`)

> One way to join two streaming tables in Spark Declarative Pipelines is by creating a materialized view that performs the join.

> This approach takes all rows from each streaming table and executes a full inner join operation and incorporates optimizations where applicable.


Use the code in section `C. Use a Materialized View to Join Two Streaming Tables`, to understand how the Gold materialized view is created.

1. This statement creates the materialized view **3_gold_db.full_order_info_gold_demo10** by joining the following streaming tables:
   - **2_silver_db.status_silver_demo10**
   - **2_silver_db.orders_silver_demo10**

2. The `COMMENT` and `TBLPROPERTIES` document the view and label it as Gold in the medallion architecture.

3. Notice that the `STREAM` keyword is not used when referencing the streaming tables when creating the materialized view.
   - This will join **all data** from both streaming tables and return your final materialized view.

### E5. Gold Materialized Views for `Cancelled` and `Delivered Orders`

Use the code in section `D. Create Gold Materialized Views for Cancelled and Delivered Orders`, to understand how the final Gold views are created.

1. This section creates two Gold materialized views from **3_gold_db.full_order_info_gold_demo10**:
   - **3_gold_db.cancelled_orders_gold_demo10**, cancelled orders with days to cancel.
   - **3_gold_db.delivered_orders_gold_demo10**, delivered orders with days to delivery.

2. Each materialized view filters on a specific status and calculates a simple business metric using `datediff`.
   - `datediff` function:
[AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/datediff) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/datediff) |
[GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/datediff)

3. The `COMMENT` and `TBLPROPERTIES` document each view and label them as Gold in the medallion architecture.


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">

- **Materialized views include built-in optimizations where applicable:**

  - Incremental refresh for materialized views: [AWS](https://docs.databricks.com/aws/en/optimizations/incremental-refresh) |[Azure](https://learn.microsoft.com/en-us/azure/databricks/optimizations/incremental-refresh) |
  [GCP](https://docs.databricks.com/gcp/en/optimizations/incremental-refresh)

  - [Spark Declarative Tables Announces New Capabilities and Performance Optimizations](https://www.databricks.com/blog/2022/06/29/delta-live-tables-announces-new-capabilities-and-performance-optimizations.html)

  - [Cost-effective, incremental ETL with serverless compute for Spark Declarative Tables pipelines](https://www.databricks.com/blog/cost-effective-incremental-etl-serverless-compute-delta-live-tables-pipelines)

- **Stateful joins (Stream to Stream):** For stateful joins in pipelines (i.e., joining incrementally as data is ingested), refer to the Optimize stateful processing in Spark Declarative Pipelines with watermarks documentation:
[AWS](https://docs.databricks.com/aws/en/dlt/stateful-processing) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/stateful-processing) |
[GCP](https://docs.databricks.com/gcp/en/dlt/stateful-processing)
   - **Stateful joins are an advanced topic and outside the scope of this course.**

  </div>
</div>




## F. Create the Production Pipeline
Follow the steps below to modify the pipeline settings and run the production pipeline.

### F1. Review and Modify the Pipeline Settings

1. Complete the following steps in your **Lakeflow Editor** to configure your Spark Declarative Pipeline for **production**:

   a. Select **Settings** to view your pipeline settings.

   b. In the **Pipeline settings** section, you can:
      - Modify the **Pipeline name** and **Run as** settings (this lab does not give you permission to modify **Run as**).

         - If you had permission, you could select the pencil icon ![pencil_settings_icon.png](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/pencil_settings_icon.png) next to **Run as** to modify the option.

         - You can optionally change the executor of the pipeline to a service principal. A service principal is an identity you create in Databricks for use with automated tools, jobs, and applications.
            - For more information, see the **What is a service principal?** documentation: [AWS](https://docs.databricks.com/aws/en/admin/users-groups/service-principals#what-is-a-service-principal) |
            [Azure](https://learn.microsoft.com/en-us/azure/databricks/admin/users-groups/service-principals#what-is-a-service-principal) |
            [GCP](https://docs.databricks.com/gcp/en/admin/users-groups/service-principals#what-is-a-service-principal)

      - In **Pipeline mode**, ensure **Triggered** is selected so the pipeline runs on a schedule to incrementally process data.
        - Alternatively, you can choose **Continuous** mode to keep the pipeline running at all times.
        - For more details, see **Triggered vs. continuous pipeline mode**: [AWS](https://docs.databricks.com/aws/en/dlt/pipeline-mode) |
        [Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/pipeline-mode) |
        [GCP](https://docs.databricks.com/gcp/en/dlt/pipeline-mode)

   c. In the **Code assets** section, confirm that:

      - **Root folder** points to this pipeline project (**10 - Deploying a Pipeline to Production Project**).

      - **Source code** references the **orders** and **status** folders within this project.

   d. In the **Default location for data assets** section, confirm the following:

      - **Default catalog** is your **labuser** catalog.

      - **Default schema** is the **default** schema.

   e. In the **Compute** section, confirm that **Serverless** compute is selected.

   f. In the **Configuration** section, ensure that the `source` key is set to your data source volume path: `/Volumes/labuser_/sdp_1_bronze/source`

2. In the **Advanced settings** section at the bottom:

   a. Expand **Advanced settings**.

   b. Click **Edit advanced settings**.

   c. For **Channel**, you can leave it as **Current** for training purposes:
    - **Current** - Uses the latest stable Databricks Runtime version, recommended for production.
    - **Preview** - Uses a more recent, potentially less stable Runtime version, ideal for testing upcoming features.
    - View the **Apache Spark™ Declarative Pipelines release notes** and the release upgrade process documentation for more information: [AWS](https://docs.databricks.com/aws/en/release-notes/dlt/) |
    [Azure](https://learn.microsoft.com/en-us/azure/databricks/release-notes/dlt/) |
    [GCP](https://docs.databricks.com/gcp/en/release-notes/dlt/)


<div style="background: #FFFDE7; border: 2px solid #FFAB00; border-radius: 8px; padding: 16px 20px; font-size: 14pt; line-height: 1.8; color: #0b2026; margin: 8px 0 8px 28px;">
  <div style="font-weight: 700; font-size: 15pt; margin-bottom: 10px;">   d. ⚠️ REQUIRED — In the <strong>Event logs</strong> section:</div>
  <ul style="margin: 0; padding-left: 20px;">
    <li>Select <strong>Publish event log to Unity Catalog</strong>.</li>
    <li><strong>Event log name</strong> - <code>event_log_demo10</code>.</li>
    <li><strong>Event log catalog</strong> - <strong>labuser</strong> catalog.</li>
    <li><strong>Event log schema</strong> - <strong>sdp_1_bronze</strong> schema.</li>
    <li>Select <strong>Save</strong>.</li>
  </ul>
  <div style="margin-top: 12px; font-size: 13.5pt; color: #5A6F77;">
    <strong>NOTE:</strong> If the event log is not saved to the correct location, the later event log exploration steps will not work properly.
  </div>
</div>

3\. Click **Save** to save your pipeline settings.

### F2. Schedule the Pipeline
1. Once your pipeline is production-ready, you'll want to **schedule it to run either on a time interval or continuously**.

   For this demonstration, we'll:
   - Schedule the pipeline to run every day at 8:00 PM.
   - Optionally configure notifications to alert you upon job **Start**, **Success**, and **Failure**.
     *(If you don't want email notifications, you can skip this step.)*

   Complete the following steps to schedule the pipeline:

   a. Select the **Schedule** button (might be a small calendar icon if your screen is minimized).

   b. For the job name, leave it as **10 - Deploying a Pipeline to Production Project - labuser-name**.

   c. Below **Job name**, select **Advanced**.

   d. In the **Schedule** section, configure the following:
   - Set the **Day**.
   - Set the time to **20:00** (8:00 PM).
   - Leave the **Timezone** as default.
   - Select **More options**, and under **Notifications**, add your email to receive alerts for:
     - **Start**
     - **Success**
     - **Failure**

   e. Click **Create** to save and schedule the job.

  **NOTE:** You could also set the pipeline to run a few minutes after your current time to see it start through the scheduler.

## G. Run and View the Spark Declarative Pipeline for `orders` and `status`

1. Manually run (instead of waiting for the job for training purposes) your Spark Declarative Pipeline and view the results.
    - **NOTE:** Currently we have one JSON file in both the **status** and **orders** volumes.

2. After the pipeline has completed its first run, complete the following:

   a. Examine the **Pipeline graph** and confirm:
      - **status flow**
         - 536 rows were read into the **status_bronze_demo10** and **status_silver_demo10** streaming tables
      - **orders flow**
         - 174 rows were read into the **orders_bronze_demo10** and **orders_silver_demo10** streaming tables
         - 7 rows are in the **gold_orders_by_date_demo10** materialized view
      - **streaming tables join and gold materialized views**
         - 536 rows are in the **full_order_info_gold_demo10** materialized view (JOIN)
            - 8 rows are in the **cancelled_orders_gold_demo10** materialized view
            - 94 rows are in the **delivered_orders_gold_demo10** materialized view


#### Checkpoint

  ![Final Demo 10 Pipeline](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/deploying-a-pipeline-to-production/demo10_pipeline_image_run1.png)

## H. Incrementally Process New Data in Your Pipeline

### H1. Land More Files in your Volume
1. Run the cell below to add **4** more JSON files to your volumes to simulate new files being landed into cloud storage:
    - `/Volumes/labuser/sdp_1_bronze/source/orders`
    - `/Volumes/labuser/sdp_1_bronze/source/status`

In [0]:
%python

## Find data in workspace data folder
data_path = "/Volumes/dbacademy/default/data"

## Land JSON files to your orders volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/orders',
    target_volume_path=f'{source_volume_path}/orders',
    n=5
)

## Land JSON files to your status volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/status',
    target_volume_path=f'{source_volume_path}/status',
    n=5
)

In [0]:
%python
orders_list = spark.sql(f"LIST '/Volumes/{my_catalog}/sdp_1_bronze/source/orders'")
status_list = spark.sql(f"LIST '/Volumes/{my_catalog}/sdp_1_bronze/source/status'")
display(orders_list)
display(status_list)

### H2. Run the Pipeline to Incrementally Process New Data
1. After you have landed **4** new files into the data source volumes, **run the pipeline to process the newly landed JSON files**.


2. After the pipeline completes, view your Pipeline run. Notice the following:

    - **status flow**
        - The **status** bronze to silver flow ingests 410 new rows.

    - **orders flow**
        - The **orders** bronze to silver flow ingests 98 new rows.
        - The **orders_by_date_gold_demo10** materialized view contains 11 rows.

    - **streaming tables join and gold materialized views**
        - The **full_order_info_gold_demo10** materialized view join contains a total of 946 rows (the previous 536 rows + the new 410 rows).
        - The **cancelled_orders_gold_demo10** materialized view contains 21 rows.
        - The **delivered_orders_gold_demo10** materialized view contains 176 rows.

#### Checkpoint - 4 New Files
![Pipeline Demo 10](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/deploying-a-pipeline-to-production/demo10_pipeline_image_run2.png)

3. In the window at the bottom of your pipeline:

    a. Select the **Expectations** link for the **status_silver_demo10** table.

    b. It should contain the value **1 met | 1 unmet**. Notice that in this run, 7.6% (31 rows) for the **valid_order_status** expectation returned a warning.

    c. This is something we would want to investigate and address in future stages of the pipeline.

## I. Introduction to the Pipeline Event Log (Advanced Topic)

After running your pipeline and successfully publishing the event log as a table named **event_log_demo10** in your **labuser.default** schema (database), begin exploring the event log.

Here we will quickly introduce the event log. **To process the event log you will need knowledge of parsing JSON formatted strings.**

  - Monitor Apache Spark™ Declarative Pipelines documentation:
[AWS](https://docs.databricks.com/aws/en/dlt/observability) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/observability) |
[GCP](https://docs.databricks.com/gcp/en/dlt/observability)

**TROUBLESHOOT:**
- **REQUIRED:** If you did not run the pipeline and publish the event log, the code below will not run. Please make sure to complete all steps before starting this section.

- **HIDDEN EVENT LOG:** By default, Spark Declarative Pipelines writes the event log to a hidden UC table in the default catalog and schema configured for the pipeline. While hidden, the table can still be queried by all sufficiently privileged users. By default, only the owner of the pipeline can query the event log table. By default, the name for the hidden event log is formatted as:

  - `catalog.schema.event_log_{pipeline_id}` - where the pipeline ID is the system-assigned UUID with dashes replaced by underscores.

  - Query the Event Log: [AWS](https://docs.databricks.com/aws/en/ldp/monitor-event-logs#query-the-event-log) |
  [Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/monitor-event-logs#query-event-log) |
  [GCP](https://docs.databricks.com/gcp/en/ldp/monitor-event-logs#query-the-event-log)

1. Complete the following steps to view the **labuser.default.event_log_demo10** event log in your catalog:

   a. Select the catalog icon ![Catalog Icon](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/catalog_icon.png) from the left navigation pane.

   b. Expand your **labuser** catalog.

   c. Expand the following schemas (databases):
      - **sdp_1_bronze**
      - **sdp_2_silver**
      - **sdp_3_gold**

   d. Notice the following:
      - In the **sdp_1_bronze**, **sdp_2_silver**, and **sdp_3_gold** schemas, the pipeline streaming tables and materialized views were created (they end with **demo10**).
      - In the **sdp_1_bronze** schema, the pipeline has published the event log as a table named **event_log_demo10**.

**NOTE:** You might need to refresh the catalogs to view the streaming tables, materialized views, and event log.


2. Query your **labuser.sdp_1_bronze.event_log_demo10** table to see what the event log looks like.

   Notice that it contains all events within the pipeline as **STRING** columns (typically JSON-formatted strings) or **STRUCT** columns. Databricks supports the `:` (colon) operator to parse JSON fields. See the `:` operator documentation: [AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/colonsign) | 
   [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/colonsign) |
   [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/colonsign)

   The following table describes the event log schema. Some fields contain JSON data—such as the **details** field—which must be parsed to perform certain queries.

In [0]:
SELECT *
FROM sdp_1_bronze.event_log_demo10;

| Field          | Description |
|----------------|-------------|
| `id`           | A unique identifier for the event log record. |
| `sequence`     | A JSON document containing metadata to identify and order events. |
| `origin`       | A JSON document containing metadata for the origin of the event, for example, the cloud provider, the cloud provider region, user_id, pipeline_id, or pipeline_type to show where the pipeline was created, either DBSQL or WORKSPACE. |
| `timestamp`    | The time the event was recorded. |
| `message`      | A human-readable message describing the event. |
| `level`        | The event type, for example, INFO, WARN, ERROR, or METRICS. |
| `maturity_level` | The stability of the event schema. The possible values are:<br><br>- **STABLE**: The schema is stable and will not change.<br>- **NULL**: The schema is stable and will not change. The value may be NULL if the record was created before the maturity_level field was added (release 2022.37).<br>- **EVOLVING**: The schema is not stable and may change.<br>- **DEPRECATED**: The schema is deprecated and the pipeline runtime may stop producing this event at any time. |
| `error`        | If an error occurred, details describing the error. |
| `details`      | A JSON document containing structured details of the event. This is the primary field used for analyzing events. |
| `event_type`   | The event type. |

**Event Log Schema:**
[AWS](https://docs.databricks.com/aws/en/ldp/monitor-event-log-schema) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/monitor-event-log-schema) |
[GCP](https://docs.databricks.com/gcp/en/ldp/monitor-event-log-schema)

3. The majority of the detailed information you will want from the event log is located in the **details** column, which is a JSON-formatted string. You will need to parse this column.

   You can find more information in the Databricks documentation on how to query JSON strings: [AWS](https://docs.databricks.com/aws/en/semi-structured/json) |
   [Azure](https://learn.microsoft.com/en-us/azure/databricks/semi-structured/json) |
   [GCP](https://docs.databricks.com/gcp/en/semi-structured/json)

   The code below will:

   - Return the **event_type** column.

   - Return the entire **details** JSON-formatted string.

   - Parse out the **flow_progress** values from the **details** JSON-formatted string, if they exist.

   - Parse out the **user_action** values from the **details** JSON-formatted string, if they exist.


In [0]:
SELECT
  id,
  event_type,
  details,
  details:flow_progress,
  details:user_action
FROM sdp_1_bronze.event_log_demo10

4. One use case for the event log is to examine data quality metrics for all runs of your pipeline. These metrics provide valuable insights into your pipeline, both in the short term and long term. Metrics are captured for each constraint throughout the entire lifetime of the table.

   Below is an example query to obtain those metrics. We won't dive into the JSON parsing code here. This example simply demonstrates what's possible with the **event_log**.

   Run the cell and observe the results. Notice the following:
   - The **passing_records** for each constraint are displayed.
   - The **failing_records** (WARN) for each constraint are displayed.

**NOTE:** If you have selected **Run pipeline with full table refresh** at any time during your pipeline, your results will include metrics from previous runs as well as from the full refresh. Additional logic is required to isolate results after the full table refresh. This is outside the scope of this course.


In [0]:
CREATE OR REPLACE TEMPORARY VIEW dq_source_vw AS
SELECT explode(
            from_json(details:flow_progress:data_quality:expectations,
                      "array<struct<name: string, dataset: string, passed_records: int, failed_records: int>>")
          ) AS row_expectations
   FROM sdp_1_bronze.event_log_demo10
   WHERE event_type = 'flow_progress';


-- View the data
SELECT
  row_expectations.dataset as dataset,
  row_expectations.name as expectation,
  SUM(row_expectations.passed_records) as passing_records,
  SUM(row_expectations.failed_records) as warnings_records
FROM dq_source_vw
GROUP BY row_expectations.dataset, row_expectations.name
ORDER BY dataset;

### Summary

This was a quick introduction to the pipeline **event_log**. With the **event_log**, you can investigate all aspects of your pipeline runs to explore the runs as well as create overall reports. Feel free to investigate the **event_log** further on your own.

## Additional Resources

- Apache Spark™ Declarative Pipelines properties reference:
[AWS](https://docs.databricks.com/aws/en/dlt/properties#dlt-table-properties) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/properties#pipeline-table-properties) |
[GCP](https://docs.databricks.com/gcp/en/dlt/properties#dlt-table-properties)

- Table properties and table options:
[AWS](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-tblproperties) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-ddl-tblproperties) |
[GCP](https://docs.databricks.com/gcp/en/sql/language-manual/sql-ref-syntax-ddl-tblproperties)

- Triggered vs. continuous pipeline mode:
[AWS](https://docs.databricks.com/aws/en/dlt/pipeline-mode) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/pipeline-mode) |
[GCP](https://docs.databricks.com/gcp/en/dlt/pipeline-mode)

- Development and production modes:
[AWS](https://docs.databricks.com/aws/en/ldp/updates#development-mode) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/updates#development-mode) |
[GCP](https://docs.databricks.com/gcp/en/ldp/updates#development-mode)

- Monitor Apache Spark™ Declarative Pipelines:
[AWS](https://docs.databricks.com/aws/en/dlt/observability) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/observability) |
[GCP](https://docs.databricks.com/gcp/en/dlt/observability)

- **Materialized views include built-in optimizations where applicable:**
  - Incremental refresh for materialized views:
[AWS](https://docs.databricks.com/aws/en/optimizations/incremental-refresh) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/optimizations/incremental-refresh) |
[GCP](https://docs.databricks.com/gcp/en/optimizations/incremental-refresh)
  - [Spark Declarative Tables Announces New Capabilities and Performance Optimizations](https://www.databricks.com/blog/2022/06/29/delta-live-tables-announces-new-capabilities-and-performance-optimizations.html)
  - [Cost-effective, incremental ETL with serverless compute for Spark Declarative Tables pipelines](https://www.databricks.com/blog/cost-effective-incremental-etl-serverless-compute-delta-live-tables-pipelines)

- **Stateful joins:** For stateful joins in pipelines (i.e., joining incrementally as data is ingested), refer to the Optimize stateful processing in Apache Spark™ Declarative Pipelines with watermarks documentation:
[AWS](https://docs.databricks.com/aws/en/dlt/stateful-processing) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/stateful-processing) |
[GCP](https://docs.databricks.com/gcp/en/dlt/stateful-processing). 
  - **Stateful joins are an advanced topic and outside the scope of this course.**


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>
